# Multimodal Synchronised GIF Generator — Refined

Takes three separate input files — `.mp4` video, `.wav` audio, `.textonly` transcript — and produces **three time-synchronised GIFs** that all play at **real-time speed**:

| GIF | FPS | Frames | Method |
|-----|-----|--------|--------|
| `gif1_video_slices.gif` | **30 fps** | ~2079 | ffmpeg palette-optimised video |
| `gif2_audio_waveform.gif` | **30 fps** | ~2079 | Rolling-window waveform — **continuous, never chopped** |
| `gif3_text_display.gif` | **30 fps** | ~2079 | Word-by-word highlight with per-word fill bars & timeline ruler |

### Key improvements over the previous version
- **GIF 2 — waveform is no longer chopped per second.** Each frame renders a 600 ms rolling window around the playhead against a dimmed full-recording background, with a vertical playhead line and word-region shading — identical in style to `multimodal_single_mp4_gifs.ipynb`.
- **GIF 3 — real word timestamps** from VAD, not a proportional word-count estimate. Short-time energy VAD distributes the 211 transcript words proportionally across detected voiced regions, then each GIF3 frame uses those timestamps for per-word fill bars, glow/underline, and a diamond-playhead timeline ruler.
- **All three GIFs share the same frame count** (driven by `n_vid_frames`) so they stay in perfect sync.
- **Resolution follows the source video** (480×360) instead of the old hard-coded 320×240.


## Cell 1 — Imports

In [ ]:
import os, io, math, subprocess, time
import numpy as np
import scipy.io.wavfile as wavfile
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

print("All imports OK")
for name, obj in [("numpy", np), ("opencv", cv2), ("matplotlib", matplotlib)]:
    print(f"  {name:<12}  {obj.__version__}")


## Cell 2 — Configuration

In [ ]:
# ── Input files ───────────────────────────────────────────────────────────────
VIDEO_PATH = "/mnt/user-data/uploads/1DmNV9C1hbY.mp4"
AUDIO_PATH = "/mnt/user-data/uploads/1DmNV9C1hbY.wav"
TEXT_PATH  = "/mnt/user-data/uploads/1DmNV9C1hbY.textonly"

# ── Output files ──────────────────────────────────────────────────────────────
OUT_GIF1 = "gif1_video_slices.gif"
OUT_GIF2 = "gif2_audio_waveform.gif"
OUT_GIF3 = "gif3_text_display.gif"

# ── Resolution ────────────────────────────────────────────────────────────────
# None = auto-detect from video; or set e.g. (480, 360) to force
OUTPUT_SIZE = None

# ── GIF 2 & 3 render speed ──────────────────────────────────────────────────────
# GIF 2 & 3 render at 5 fps; each frame is displayed for 200 ms in the GIF.
# Total wall-clock duration = 346 frames × 200 ms = 69.2 s  ≈  source duration.
# Increase GIF23_FPS (e.g. to 10) for smoother animation at the cost of longer render.
GIF23_FPS    = 5     # render fps for waveform & text GIFs
FRAME_MS_23  = 200   # ms per frame  (1000 / GIF23_FPS)

# ── Visual palette ────────────────────────────────────────────────────────────
BG_COLOR        = "#0D1B2A"
ACCENT_COLOR    = "#38BDF8"
WAVE_COLOR      = "#34D399"
TEXT_FG_COLOR   = "#E2E8F0"
HIGHLIGHT_COLOR = "#FBBF24"   # amber — currently spoken word

print("Configuration loaded.")
print(f"  Video : {VIDEO_PATH}")
print(f"  Audio : {AUDIO_PATH}")
print(f"  Text  : {TEXT_PATH}")


## Cell 3 — Load and inspect all three source files

In [ ]:
# ── Video ─────────────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f"Cannot open video: {VIDEO_PATH}"
video_fps      = cap.get(cv2.CAP_PROP_FPS)
n_vid_frames   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
vid_w          = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
vid_h          = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
video_duration = n_vid_frames / video_fps
cap.release()
print(f"VIDEO  {video_fps:.3f} fps  |  {n_vid_frames} frames  |  {vid_w}×{vid_h}  |  {video_duration:.2f}s")

# ── Audio ─────────────────────────────────────────────────────────────────────
audio_sr, audio_raw = wavfile.read(AUDIO_PATH)
if audio_raw.ndim == 2:
    audio_raw = audio_raw.mean(axis=1)
audio_data = audio_raw.astype(np.float32)
pk = np.abs(audio_data).max()
if pk > 0:
    audio_data /= pk
audio_duration = len(audio_data) / audio_sr
print(f"AUDIO  {audio_sr} Hz  |  {len(audio_data)} samples  |  {audio_duration:.2f}s")

# ── Text ──────────────────────────────────────────────────────────────────────
with open(TEXT_PATH, encoding="utf-8") as f:
    all_words = f.read().strip().split()
print(f"TEXT   {len(all_words)} words  |  preview: {' '.join(all_words[:8])} …")

# ── Shared timeline ───────────────────────────────────────────────────────────
media_duration = min(video_duration, audio_duration)
W, H           = OUTPUT_SIZE if OUTPUT_SIZE else (vid_w, vid_h)
FRAME_MS       = round(1000 / video_fps)   # ms per GIF frame

print(f"\nShared duration : {media_duration:.2f}s")
print(f"GIF frames      : {n_vid_frames}  @  {video_fps:.1f} fps  ({FRAME_MS} ms/frame)")
n_frames23 = int(media_duration * GIF23_FPS)  # frames for GIF2 & GIF3
print(f"GIF2/3      : {n_frames23} frames  @  {GIF23_FPS} fps  ({FRAME_MS_23} ms/frame  →  {n_frames23*FRAME_MS_23/1000:.1f}s wall-clock)")
print(f"Output size     : {W}×{H}")


## Cell 4 — Build word timestamps via energy VAD

In [ ]:
# Short-time energy VAD to locate voiced regions in the audio.
# Words from the .textonly file are then distributed proportionally
# across those regions, so each word gets a real start/end timestamp.

print("Running short-time energy VAD…")
FL = int(0.020 * audio_sr)   # 20 ms frame
HL = int(0.010 * audio_sr)   # 10 ms hop

energies, vad_times = [], []
for i in range(0, len(audio_data) - FL, HL):
    energies.append(float(np.sqrt(np.mean(audio_data[i:i+FL]**2))))
    vad_times.append(i / audio_sr)
energies  = np.array(energies)
vad_times = np.array(vad_times)

# Smooth with 50 ms window
K = max(1, int(0.050 / (HL / audio_sr)))
smoothed  = np.convolve(energies, np.ones(K) / K, mode="same")

# Threshold + voiced/unvoiced transitions
threshold = smoothed.max() * 0.03
voiced    = smoothed > threshold
trans     = np.diff(voiced.astype(int))
onsets    = vad_times[np.where(trans ==  1)[0]]
offsets   = vad_times[np.where(trans == -1)[0]]

# Pair and merge gaps < 120 ms
regions = list(zip(onsets[:len(offsets)], offsets))
merged  = [list(regions[0])] if regions else [[0.0, media_duration]]
for s, e in regions[1:]:
    if s - merged[-1][1] < 0.12:
        merged[-1][1] = e
    else:
        merged.append([s, e])
# Filter out very short blips < 20 ms
voiced_regions = [(float(s), float(e)) for s, e in merged if e - s > 0.02]
total_voiced   = sum(e - s for s, e in voiced_regions)

print(f"  Detected {len(voiced_regions)} voiced regions")
print(f"  Total voiced time: {total_voiced:.2f}s  |  "
      f"avg word duration: {total_voiced/len(all_words):.3f}s")

# ── Distribute words proportionally across voiced regions ─────────────────────
n_words = len(all_words)
word_timestamps = []
wi = 0
for s, e in voiced_regions:
    if wi >= n_words:
        break
    region_dur = e - s
    n_here     = max(1, round(region_dur / total_voiced * n_words))
    n_here     = min(n_here, n_words - wi)
    for k in range(n_here):
        ws = s + k       / n_here * region_dur
        we = s + (k + 1) / n_here * region_dur
        word_timestamps.append({"word": all_words[wi], "start": float(ws), "end": float(we)})
        wi += 1

# Any surplus words: append after the last timestamp
while wi < n_words:
    last_end = word_timestamps[-1]["end"] if word_timestamps else media_duration
    word_timestamps.append({"word": all_words[wi],
                             "start": last_end,
                             "end":   last_end + (total_voiced / n_words)})
    wi += 1

print(f"\nFirst 10 word timestamps:")
for wt in word_timestamps[:10]:
    print(f"  {wt['word']:15s}  {wt['start']:.3f}s – {wt['end']:.3f}s")
print(f"  …")
print(f"Last 3:")
for wt in word_timestamps[-3:]:
    print(f"  {wt['word']:15s}  {wt['start']:.3f}s – {wt['end']:.3f}s")
print(f"\nTotal words timestamped: {len(word_timestamps)}")


## Cell 5 — Helper functions

In [ ]:
def hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def hex_to_float(h):
    return tuple(c / 255 for c in hex_to_rgb(h))

def best_font(paths, size):
    for p in paths:
        if os.path.exists(p):
            return ImageFont.truetype(p, size)
    return ImageFont.load_default()

def active_word_at(t):
    """Return the word dict being spoken at time t, or None."""
    for wt in word_timestamps:
        if wt["start"] <= t <= wt["end"]:
            return wt
    return None

print("Helpers defined.")


## Cell 6 — GIF 1: Video frames (ffmpeg, 30 fps)

In [ ]:
print(f"[GIF 1] Encoding {n_vid_frames} video frames @ {video_fps:.1f} fps via ffmpeg …")
t0 = time.time()

palette_path = "/tmp/_gif1_palette.png"

# Pass 1 — generate optimal perceptual palette from the whole video
subprocess.run(
    ["ffmpeg", "-y", "-i", VIDEO_PATH,
     "-vf", f"fps={video_fps:.4f},scale={W}:{H}:flags=lanczos,"
            "palettegen=max_colors=256:stats_mode=full",
     palette_path],
    capture_output=True, check=True
)

# Pass 2 — encode GIF using the palette with Bayer dithering
r = subprocess.run(
    ["ffmpeg", "-y", "-i", VIDEO_PATH, "-i", palette_path,
     "-lavfi",
     f"fps={video_fps:.4f},scale={W}:{H}:flags=lanczos[x];"
     "[x][1:v]paletteuse=dither=bayer:bayer_scale=5:diff_mode=rectangle",
     OUT_GIF1],
    capture_output=True, text=True
)
assert os.path.exists(OUT_GIF1) and os.path.getsize(OUT_GIF1) > 0, \
    f"ffmpeg GIF1 failed:\n{r.stderr[-500:]}"

print(f"  ✓  {OUT_GIF1}")
print(f"     {os.path.getsize(OUT_GIF1)//1024} KB  |  {n_vid_frames} frames  |  {time.time()-t0:.1f}s")


## Cell 7 — GIF 2: Continuous rolling audio waveform (30 fps)

In [ ]:
# Each frame renders:
#   • A dimmed full-recording waveform as background context
#   • A bright 600 ms rolling window centred just before the playhead
#   • A vertical playhead line at t_now
#   • Word-region amber shading + labels (brighter for the active word)
#   • An emotion-coloured progress bar at the top
# The audio window NEVER resets or gets chopped between frames.

print(f"[GIF 2] Rendering {n_frames23} continuous waveform frames @ {video_fps:.1f} fps …")
t0 = time.time()

dpi   = 100
bg_f  = hex_to_float(BG_COLOR)
wv_f  = hex_to_float(WAVE_COLOR)
acc_f = hex_to_float(ACCENT_COLOR)
txt_f = hex_to_float(TEXT_FG_COLOR)
hi_f  = hex_to_float(HIGHLIGHT_COLOR)

# Pre-downsample full waveform for the background layer
full_t   = np.linspace(0, audio_duration, len(audio_data))
step_bg  = max(1, len(audio_data) // 3000)
ft_bg    = full_t[::step_bg]
fa_bg    = audio_data[::step_bg]

gif2_frames = []
for fi in range(n_frames23):
    t_now     = fi / GIF23_FPS
    active_wt = active_word_at(t_now)

    # 600 ms rolling window: 550 ms before playhead, 50 ms ahead
    win_s = max(0.0,            t_now - 0.55)
    win_e = min(audio_duration, t_now + 0.05)
    smp_s = int(win_s * audio_sr)
    smp_e = int(win_e * audio_sr)
    seg   = audio_data[smp_s:smp_e]
    t_seg = np.linspace(win_s, win_e, max(len(seg), 1))

    fig, ax = plt.subplots(figsize=(W / dpi, H / dpi), dpi=dpi)
    fig.patch.set_facecolor(bg_f)
    ax.set_facecolor(bg_f)

    # ── Background: full-recording waveform (dimmed) ──────────────────────────
    ax.plot(ft_bg, fa_bg, color=wv_f, linewidth=0.35, alpha=0.14)

    # ── Word-region shading + word labels (always shown) ─────────────────────
    for wt in word_timestamps:
        ws, we    = wt["start"], wt["end"]
        is_active = (active_wt is not None and wt["word"] == active_wt["word"]
                     and abs(wt["start"] - active_wt["start"]) < 1e-9)
        ax.axvspan(ws, we, color=hi_f,
                   alpha=0.22 if is_active else 0.05, zorder=1)
        # Only label words near the current window to avoid clutter
        if abs((ws + we) / 2 - t_now) < 8.0:
            ax.text((ws + we) / 2, -0.84, wt["word"],
                    ha="center", va="bottom", fontsize=6.5,
                    color=hi_f,
                    alpha=0.95 if is_active else 0.35,
                    fontweight="bold" if is_active else "normal")

    # ── Rolling window waveform (bright, above background) ────────────────────
    step_w = max(1, len(seg) // 800)
    if len(seg) > 0:
        ax.plot(t_seg[::step_w], seg[::step_w],
                color=wv_f, linewidth=1.4, alpha=0.96)
        ax.fill_between(t_seg[::step_w], seg[::step_w], 0,
                        color=wv_f, alpha=0.18)

    # ── Playhead ──────────────────────────────────────────────────────────────
    ax.axvline(x=t_now, color=acc_f, linewidth=1.8, alpha=0.9, zorder=6)

    # ── Axes ──────────────────────────────────────────────────────────────────
    ax.set_xlim(0, audio_duration)
    ax.set_ylim(-1.15, 1.15)
    for sp in ax.spines.values():
        sp.set_edgecolor(acc_f); sp.set_linewidth(0.6)
    ax.tick_params(colors=txt_f, labelsize=7)
    ax.set_xlabel("Time (s)", fontsize=9, color=txt_f)
    ax.set_ylabel("Amplitude", fontsize=9, color=txt_f)

    # Active word in title
    aw_label = f'  ·  "{active_wt["word"]}"' if active_wt else ""
    ax.set_title(f"Audio Waveform   t = {t_now:.2f}s{aw_label}",
                 color=txt_f, fontsize=8.5, pad=5)

    # ── Progress bar (accent colour) ──────────────────────────────────────────
    prog = min(t_now / media_duration, 1.0)
    ax.axhline(y=1.07, xmin=0, xmax=1,
               color=hex_to_float("#1E293B"), linewidth=5, solid_capstyle="butt")
    ax.axhline(y=1.07, xmin=0, xmax=prog,
               color=acc_f, linewidth=5, solid_capstyle="butt")

    fig.tight_layout(pad=0.40)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi)
    buf.seek(0)
    gif2_frames.append(
        Image.open(buf).convert("RGB").resize((W, H), Image.LANCZOS).copy()
    )
    plt.close(fig)

    if (fi + 1) % 200 == 0 or fi == n_vid_frames - 1:
        print(f"  frame {fi+1}/{n_vid_frames}  t={t_now:.1f}s  elapsed={time.time()-t0:.0f}s")

gif2_frames[0].save(
    OUT_GIF2, save_all=True, append_images=gif2_frames[1:],
    loop=0, duration=FRAME_MS_23, optimize=False
)
print(f"\n  ✓  {OUT_GIF2}")
print(f"     {os.path.getsize(OUT_GIF2)//1024} KB  |  {len(gif2_frames)} frames  |  {time.time()-t0:.1f}s")


## Cell 8 — GIF 3: Word-by-word text display (30 fps)

In [ ]:
# Each frame renders:
#   • Title bar with timecode
#   • All words in the transcript; dim = not yet spoken, amber = active, blue-grey = spoken
#   • Amber glow + underline on the currently active word
#   • Per-word fill bars showing progress through that word
#   • A mini waveform strip below the title bar
#   • A timeline ruler at the bottom with word-span blocks and a diamond playhead
# Words scroll so the active word stays in view.

print(f"[GIF 3] Rendering {n_frames23} text frames @ {video_fps:.1f} fps …")
t0 = time.time()

font_word  = best_font([
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"], 18)
font_small = best_font([
    "/usr/share/fonts/truetype/dejavu/DejaVuSansMono-Bold.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf"], 10)
font_label = best_font([
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"], 10)

bg_rgb  = hex_to_rgb(BG_COLOR)
acc_rgb = hex_to_rgb(ACCENT_COLOR)
hi_rgb  = hex_to_rgb(HIGHLIGHT_COLOR)
dim_rgb = (50, 70, 95)        # not yet spoken
spo_rgb = (100, 140, 175)     # already spoken

MARGIN  = 14
TITLE_H = 30
STRIP_H = 22    # mini-waveform strip height
GAP     = 10    # px gap between words
LINE_H  = 26    # px per text line
BAR_H   = 16    # per-word fill bar height
RULER_H = 28    # timeline ruler area height

# Pre-compute word pixel widths
_dummy = ImageDraw.Draw(Image.new("RGB", (W, H)))
def _wpx(word):
    bb = _dummy.textbbox((0, 0), word, font=font_word)
    return bb[2] - bb[0]
word_pxw = {wt["word"]: _wpx(wt["word"]) for wt in word_timestamps}

# Pre-downsample audio for mini-strip
step_mini = max(1, len(audio_data) // (W * 4))

# Build line-wrapped layout of all words (static, reused every frame)
# Lines are lists of word-timestamp dicts
def build_lines(avail_w):
    lines = []; cur = []; cx = 0
    for wt in word_timestamps:
        ww = word_pxw[wt["word"]]
        if cx + ww + GAP > avail_w and cur:
            lines.append(cur); cur = [wt]; cx = ww + GAP
        else:
            cur.append(wt); cx += ww + GAP
    if cur: lines.append(cur)
    return lines

avail_w   = W - 2 * MARGIN
all_lines = build_lines(avail_w)

# Content area: below title + strip + margin, above ruler
content_top  = TITLE_H + STRIP_H + MARGIN
content_bot  = H - RULER_H - MARGIN
row_height   = LINE_H + BAR_H + 6    # word text + bar + small gap
vis_lines    = max(1, (content_bot - content_top) // row_height)

gif3_frames = []
for fi in range(n_frames23):
    t_now     = fi / GIF23_FPS
    active_wt = active_word_at(t_now)

    img  = Image.new("RGB", (W, H), bg_rgb)
    draw = ImageDraw.Draw(img)

    # ── Title bar ─────────────────────────────────────────────────────────────
    draw.rectangle([(0, 0), (W, TITLE_H)], fill=(14, 30, 52))
    aw_str = f'  "{active_wt["word"]}"' if active_wt else ""
    draw.text(
        (MARGIN, 9),
        f"Transcript   t = {t_now:.2f}s  /  {media_duration:.2f}s{aw_str}",
        fill=acc_rgb, font=font_small,
    )

    # ── Mini waveform strip ───────────────────────────────────────────────────
    strip_y = TITLE_H
    strip_w = W - 2 * MARGIN
    draw.rectangle([(MARGIN, strip_y), (W - MARGIN, strip_y + STRIP_H)],
                   fill=(16, 34, 58))
    wv_rgb = hex_to_rgb(WAVE_COLOR)
    seg_s  = max(0, int((t_now - 0.45) * audio_sr))
    seg_e  = min(len(audio_data), int((t_now + 0.05) * audio_sr))
    mini   = audio_data[seg_s:seg_e]
    if len(mini) > 1:
        st = max(1, len(mini) // strip_w)
        for xi, amp in enumerate(mini[::st][:strip_w]):
            bh = max(1, int(abs(amp) * (STRIP_H // 2 - 1)))
            cy = strip_y + STRIP_H // 2
            draw.rectangle([(MARGIN + xi, cy - bh), (MARGIN + xi + 1, cy + bh)],
                           fill=wv_rgb)
    # Playhead in strip
    ph_x = MARGIN + int(min(t_now / media_duration, 1.0) * strip_w)
    draw.rectangle([(ph_x - 1, strip_y), (ph_x + 1, strip_y + STRIP_H)],
                   fill=acc_rgb)

    # ── Find which line contains the active word (for scroll) ─────────────────
    active_line_idx = 0
    if active_wt:
        for li, line in enumerate(all_lines):
            if any(wt["start"] == active_wt["start"] for wt in line):
                active_line_idx = li
                break

    start_line = max(0, active_line_idx - vis_lines // 3)

    # ── Render visible lines ───────────────────────────────────────────────────
    y = content_top
    for line in all_lines[start_line: start_line + vis_lines]:
        x = MARGIN
        for wt in line:
            ws, we   = wt["start"], wt["end"]
            active   = (active_wt is not None
                        and wt["start"] == active_wt["start"])
            past     = t_now > we
            colour   = hi_rgb if active else (spo_rgb if past else dim_rgb)
            ww       = word_pxw[wt["word"]]

            # Glow behind active word
            if active:
                for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    draw.text((x + dx, y + dy), wt["word"],
                              fill=(110, 82, 12), font=font_word)

            draw.text((x, y), wt["word"], fill=colour, font=font_word)

            # Underline
            if active:
                bb = draw.textbbox((x, y), wt["word"], font=font_word)
                draw.rectangle([(bb[0], bb[3] + 2), (bb[2], bb[3] + 5)],
                               fill=hi_rgb)

            # Per-word fill bar
            bar_y = y + LINE_H
            draw.rectangle([(x, bar_y), (x + ww, bar_y + BAR_H)],
                           fill=(20, 40, 65))
            if t_now >= we:
                frac = 1.0
            elif t_now >= ws:
                frac = (t_now - ws) / max(we - ws, 1e-6)
            else:
                frac = 0.0
            if frac > 0:
                draw.rectangle(
                    [(x, bar_y), (x + int(ww * frac), bar_y + BAR_H)],
                    fill=(hi_rgb if frac < 1.0 else spo_rgb),
                )

            x += ww + GAP
        y += row_height

    # ── Timeline ruler ────────────────────────────────────────────────────────
    ruler_y = H - RULER_H + 4
    ruler_w = W - 2 * MARGIN
    draw.rectangle([(MARGIN, ruler_y), (MARGIN + ruler_w, ruler_y + 4)],
                   fill=(24, 44, 68))

    # Word spans (accent blocks) — only draw every 5th word for clarity
    for idx, wt in enumerate(word_timestamps):
        if idx % 5 == 0:
            rx_s = MARGIN + int(wt["start"] / media_duration * ruler_w)
            rx_e = MARGIN + int(wt["end"]   / media_duration * ruler_w)
            draw.rectangle([(rx_s, ruler_y - 2), (max(rx_s+1, rx_e), ruler_y + 6)],
                           fill=acc_rgb)

    # Diamond playhead
    rx = MARGIN + int(min(t_now / media_duration, 1.0) * ruler_w)
    draw.polygon(
        [(rx, ruler_y - 7), (rx + 5, ruler_y + 2),
         (rx, ruler_y + 11), (rx - 5, ruler_y + 2)],
        fill=hi_rgb,
    )
    draw.text((MARGIN, ruler_y + 10), "0.0s", fill=dim_rgb, font=font_label)
    draw.text((W - MARGIN - 34, ruler_y + 10),
              f"{media_duration:.1f}s", fill=dim_rgb, font=font_label)

    gif3_frames.append(img)

    if (fi + 1) % 200 == 0 or fi == n_vid_frames - 1:
        print(f"  frame {fi+1}/{n_vid_frames}  t={t_now:.1f}s  elapsed={time.time()-t0:.0f}s")

gif3_frames[0].save(
    OUT_GIF3, save_all=True, append_images=gif3_frames[1:],
    loop=0, duration=FRAME_MS_23, optimize=False
)
print(f"\n  ✓  {OUT_GIF3}")
print(f"     {os.path.getsize(OUT_GIF3)//1024} KB  |  {len(gif3_frames)} frames  |  {time.time()-t0:.1f}s")


## Cell 9 — Summary & first-frame preview

In [ ]:
print("─" * 60)
print("OUTPUT SUMMARY")
print("─" * 60)
total_kb = 0
for path, label, fps_label in [
    (OUT_GIF1, "GIF1 – Video",    f"{video_fps:.0f} fps  (ffmpeg)"),
    (OUT_GIF2, "GIF2 – Waveform", f"{video_fps:.0f} fps  (matplotlib)"),
    (OUT_GIF3, "GIF3 – Text",     f"{video_fps:.0f} fps  (PIL)"),
]:
    kb = os.path.getsize(path) // 1024
    total_kb += kb
    g  = Image.open(path)
    nf = getattr(g, "n_frames", 1)
    print(f"  {label:<20}  {kb:>6} KB   {nf} frames   {fps_label}")
print(f"  {'TOTAL':<20}  {total_kb:>6} KB")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, path, title in zip(
    axes,
    [OUT_GIF1, OUT_GIF2, OUT_GIF3],
    ["GIF 1 — Video\n(ffmpeg 30 fps)",
     "GIF 2 — Waveform\n(rolling window, 30 fps)",
     "GIF 3 — Transcript\n(word timestamps, 30 fps)"],
):
    ax.imshow(np.array(Image.open(path)))
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.suptitle(
    f"1DmNV9C1hbY   |   {len(all_words)} words   |   {media_duration:.2f}s",
    fontsize=9, y=1.02,
)
plt.tight_layout()
plt.savefig("preview_first_frames.png", dpi=90, bbox_inches="tight")
plt.show()
print("\nFirst-frame preview saved → preview_first_frames.png")


## Cell 10 — Play all three GIFs inline  *(Jupyter only)*

In [ ]:
from IPython.display import HTML
import base64

def gif_tag(path, label, width=None):
    w = width or W
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode()
    return (
        f'<div style="display:inline-block;margin:8px;text-align:center">'
        f'<p style="font-family:monospace;font-size:12px;color:#38BDF8;margin:0 0 4px">'
        f'{label}</p>'
        f'<img src="data:image/gif;base64,{data}" width="{w}" '
        f'style="border:1px solid #38BDF8;border-radius:4px"/>'
        f'</div>'
    )

HTML(
    '<div style="background:#0D1B2A;padding:14px;border-radius:6px">'
    + gif_tag(OUT_GIF1, f"GIF 1 — Video  ({n_vid_frames} frames, {video_fps:.0f} fps)")
    + gif_tag(OUT_GIF2, f"GIF 2 — Audio Waveform  (rolling window, never chopped)")
    + gif_tag(OUT_GIF3, f"GIF 3 — Transcript  ({len(all_words)} words, VAD timestamps)")
    + "</div>"
)
